# Agent Wordle

## Same harness, new game

In `setup/setup.ipynb` we built an agent that finds its way out of a maze with one tool and a loop. Here the same harness plays Wordle.

The secret word is a five-letter travel word. The agent has six guesses. After each guess, the tool says which letters are in the right place, which are in the word but somewhere else, and which are not in the word at all.

The model never sees the secret word. It only sees what the tool tells it, so every guess has to be worked out from the clues.

Everything runs on free models, with the OpenRouter key in your `.env` file from the setup notebook.

In [ ]:
import json
import os
import random
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("No OpenRouter API key found. Please check your .env file (see setup/setup.ipynb).")

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(
    base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5
)

## The game

The secret words are all things a travel insurer worries about.

`score_guess` marks each letter the way Wordle does: G for the right place, Y for in the word but somewhere else, and - for not in the word. A repeated letter only gets a Y as many times as it appears in the secret word.

In [ ]:
TRAVEL_WORDS = [
    "PLANE", "BEACH", "CLAIM", "HOTEL", "VISAS", "FERRY", "TRAIN", "SCUBA", "DELAY", "TOURS",
    "HIKER", "GUIDE", "MOTEL", "OASIS", "ROUTE", "SKIER", "STORM", "COAST", "CABIN", "TAXIS",
]
MAX_GUESSES = 6


def score_guess(guess, secret):
    """Return one mark per letter: G for the right place, Y for somewhere else, - for not in the word"""
    marks = ["-"] * 5
    unmatched = []
    for i, (g, s) in enumerate(zip(guess, secret)):
        if g == s:
            marks[i] = "G"
        else:
            unmatched.append(s)
    for i, g in enumerate(guess):
        if marks[i] == "-" and g in unmatched:
            marks[i] = "Y"
            unmatched.remove(g)
    return "".join(marks)


print(score_guess("CRANE", "BEACH"))
print(score_guess("EERIE", "HOTEL"))
print(score_guess("BEACH", "BEACH"))

`EERIE` against `HOTEL` gets a single Y: there is only one E in `HOTEL`, so only one of the three E's in the guess counts.

## Part 1 of the harness: the tool

The game keeps its state in a small dictionary: the secret word and the guesses so far. `new_game` picks a word at random.

`guess_word` checks the guess, scores it and describes the result in words, since small models follow sentences more reliably than a row of symbols. Once the game is over, it says so.

In [ ]:
game = {}


def new_game(secret=None):
    """Start a new game with a random travel word, or the one given"""
    game["secret"] = (secret or random.choice(TRAVEL_WORDS)).upper()
    game["guesses"] = []
    game["won"] = False
    print("New game: the secret word has 5 letters")


def game_over():
    return game["won"] or len(game["guesses"]) >= MAX_GUESSES


def guess_word(word):
    """Score a guess at the secret word and describe the result"""
    guess = word.strip().upper()
    print(f"Guess: {guess}")
    if game_over():
        return "The game is over. Please stop guessing and report the result."
    if len(guess) != 5 or not guess.isalpha():
        return "Please guess a single word of exactly 5 letters. This attempt was not counted."

    marks = score_guess(guess, game["secret"])
    game["guesses"].append((guess, marks))
    print(f"       {marks}")

    if marks == "GGGGG":
        game["won"] = True
        return f"Correct! The secret word is {guess}. You solved it in {len(game['guesses'])} guesses."

    descriptions = {
        "G": "is in the word and in the right place",
        "Y": "is in the word but in a different place",
        "-": "is not in the word",
    }
    lines = [f"Position {i + 1}: {g} {descriptions[m]}" for i, (g, m) in enumerate(zip(guess, marks))]
    left = MAX_GUESSES - len(game["guesses"])
    if left == 0:
        lines.append(f"That was the last guess. The secret word was {game['secret']}.")
    else:
        lines.append(f"You have {left} guesses left.")
    return "\n".join(lines)

Try it by hand first. We choose the secret word ourselves so we know what to expect.

In [ ]:
new_game("BEACH")
print(guess_word("CRANE"))

As with the maze, the model never sees our Python code. It sees this description of the tool.

In [ ]:
guess_word_json = {
    "type": "function",
    "function": {
        "name": "guess_word",
        "description": "Guess the secret 5-letter word. Returns feedback on each letter of the guess.",
        "parameters": {
            "type": "object",
            "properties": {
                "word": {
                    "type": "string",
                    "description": "A 5-letter English word",
                },
            },
            "required": ["word"],
        },
    },
}

tools = [guess_word_json]


def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = guess_word(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": result}

## Part 2 of the harness: the loop

This is the maze's loop with one addition: once the game is over, the harness tells the model, so it reports the result instead of guessing on.

The system prompt asks the agent to sum up what it knows before each guess. Writing the clues down helps a small model keep track of them.

In [ ]:
SYSTEM_PROMPT = """
You are playing Wordle. The secret word is a 5-letter English word related to travel.
Use the guess_word tool to make one guess at a time. You have 6 guesses.
Before each guess, briefly say what you know so far: letters in the right place, letters elsewhere in the word, and letters ruled out.
Every new guess must fit all of the clues so far.
When the game ends, report the result in a sentence or two. Do not use emojis.
"""


def run_agent(task, max_steps=15):
    """Call the model in a loop, running the tool whenever it asks, until it reports the result"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
            if game_over():
                messages.append({"role": "user", "content": "The game is over. Please report the result."})
        elif text:
            return
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

## Play

A game takes a minute or two and uses up to about 10 requests. If you see an error, run the cell again.

In [ ]:
new_game()
run_agent("Let's play Wordle! Make your first guess.")

## The scoreboard

In [ ]:
for guess, marks in game["guesses"]:
    print(" ".join(guess))
    print(" ".join(marks))
    print()

result = "won" if game["won"] else "lost"
print(f"The agent {result}. Guesses used: {len(game['guesses'])}. The word was {game['secret']}.")

## Try it yourself

- Play again. Does the agent ever break its own clues, such as guessing a letter it has already ruled out?
- Change `MODEL` to one named free model from https://openrouter.ai/models?max_price=0 and see which one plays best.
- Take out the line in `SYSTEM_PROMPT` that asks the agent to sum up what it knows before each guess. Does it play worse? That one line is a small piece of harness engineering.
- Play against the agent: call `new_game()`, then `guess_word("...")` yourself, and compare your score with the agent's.

A free OpenRouter account allows 50 requests a day to free models.